# **Investment Portfolio Management**

## **About the scenario**
This scenario demonstrates best practices in a data engineering context, where notebooks are used to orchestrate complex tasks involving real-time data fetching, computation, and report delivery. The notebook is modular, and secure, employing batch processing, error handling, and proper separation of concerns.

In this scenario, we will:

1. Ingest a CSV file containing the user’s investment portfolio.
2. Fetch real-time stock prices via an API using *Function Calling*.
3. Perform calculations on the portfolio, including total value and percentage changes, using *Code Interpreter*.
4. Generate a detailed report summarizing the portfolio's performance.
5. Save the report to Azure Blob Storage using *Function Calling*.

## **Time**
You should expect to spend 10-15 minutes building and running this scenario. 

## **Before you begin**

### Install required libraries

In [ ]:
# Install the packages
%pip install -r ./requirements.txt

### Set parameters

### Setup credentials

## **Ingest data**
This scenario ingests data from two separte sources:
- Files from the folder [`data/`](./data/) in this repo. This file simulates a personal investment portfolio with ticker `Symbol`, `Average_Cost`, `Quantity` data. You can clone this repo or copy this folder to make sure you have access to these files when running this scenario.
- Latest stock prices by ticker symbol using Yahoo Finance ([`yfinance`](https://pypi.org/project/yfinance/)).

### Personal Investment Portfolio Data

- ==NOTE: find bigger better dataset with wget and get table==

In [18]:
import pandas as pd
from pandas.errors import EmptyDataError

# Load the investment portfolio CSV file
def load_portfolio_csv(file_path: str) -> pd.DataFrame:
    try:
        portfolio_df = pd.read_csv(file_path)
    except EmptyDataError:
        portfolio_df = pd.DataFrame()
    return portfolio_df

### Get Latest Stock Prices
This function retrieves the stock data for a specified `ticker` symbol using the yfinance library, specifically pulling the latest data for the last trading day.

In [19]:
import yfinance as yf

def fetch_stock_price(ticker_symbol: str) -> dict:
    """
    Fetch the latest stock price and opening price for a given ticker symbol.

    Parameters:
    - ticker_symbol (str): The ticker symbol of the stock to retrieve data for.

    Returns:
    - dict: A dictionary containing the following keys:
        - "company_name" (str): The name of the company corresponding to the ticker symbol.
        - "ticker" (str): The ticker symbol provided as input.
        - "open_price" (float): The opening price of the stock for the latest trading day.
        - "latest_close_price" (float): The closing price of the stock for the latest trading day.
        - "error" (str): An error message if an issue occurred, or if no data was found.
        
    Example:
    >>> fetch_stock_price("AAPL")
    {'ticker': 'AAPL', 'open_price': 145.3, 'latest_close_price': 148.9}
    """
    
    # Define the default return structure
    default_response = {
        "company_name": None,
        "ticker": ticker_symbol,
        "latest_open_price": None,
        "latest_close_price": None,
        "error": None
    }

    try:
        # Fetch the stock's trading history for the last day
        stock = yf.Ticker(ticker_symbol)
        stock_data = stock.history(period="1d")

        # Check if the data is empty, indicating an invalid ticker or no data available
        if stock_data.empty:
            return {"error": f"No data found for ticker symbol: {ticker_symbol}"}

         # Extract the company name from the 'info' dictionary
        company_name = stock.info.get('longName', "Company name not available")

        # Extract the latest available open and close prices
        latest_open_price = stock_data['Open'].iloc[-1]
        latest_close_price = stock_data['Close'].iloc[-1]
        
        return {
            "company_name": company_name,
            "ticker": ticker_symbol,
            "latest_open_price": latest_open_price,
            "latest_close_price": latest_close_price,
            "error": None
        }

    except KeyError as e:
        return {**default_response, "error": f"Data missing for key: {e}. Verify the ticker symbol."}

    except EmptyDataError as e:
        return {**default_response, "error": f"yfinance error or data fetch issue: {e}"}

    except Exception as e:
        return {**default_response, "error": f"Unexpected error: {type(e).__name__}: {e}"}

## **Transformation**

--- add some error ticker
-- normalize

In [36]:
# Define function to cleanse and format the stock prices dataframe
def cleanse_stock_prices(stock_prices_df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleanse and format the stock prices dataframe.

    Parameters:
    - stock_prices_df (pd.DataFrame): The dataframe containing stock prices data.

    Returns:
    - pd.DataFrame: A new dataframe containing the cleansed and formatted data.
    """
    try:
        
        # Remove rows with missing or non-positive prices
        #stock_prices_df = stock_prices_df.dropna()
        stock_prices_df = stock_prices_df[stock_prices_df['latest_close_price'] > 0]

        # Drop any rows where the 'error' column is not null
        stock_prices_df = stock_prices_df[stock_prices_df['error'].isnull()]

        # Ensure the company name is in title case
        stock_prices_df['company_name'] = stock_prices_df['company_name'].apply(lambda x: x.title() if x else "Company name not available")

        # Ensure the ticker symbol is in uppercase
        stock_prices_df['ticker'] = stock_prices_df['ticker'].str.upper()

        # Round the prices to 2 decimal places
        stock_prices_df['latest_open_price'] = stock_prices_df['latest_open_price'].round(2)  
        stock_prices_df['latest_close_price'] = stock_prices_df['latest_close_price'].round(2)

        # Drop the 'error' column
        stock_prices_df = stock_prices_df.drop(columns=['error'])

        # Reset the index of the dataframe after removing rows and columns to ensure it is continuous
        stock_prices_df = stock_prices_df.reset_index(drop=True)
        
        return stock_prices_df

    except KeyError as key_error:
        print(f"KeyError: {key_error}")
        return pd.DataFrame()  # Return an empty DataFrame in case of error

    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in case of error


In [28]:
# Define function to cleanse and format portfolio data and merge with stock prices
def cleanse_and_merge_data(portfolio_df: pd.DataFrame, stock_prices_df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleanse and merge the portfolio and stock prices dataframes.

    Parameters:
    - portfolio_df (pd.DataFrame): The dataframe containing the portfolio data.
    - stock_prices_df (pd.DataFrame): The dataframe containing stock prices data.

    Returns:
    - pd.DataFrame: A new dataframe containing the cleansed and merged data.
    """
    try:
        # Drop any rows in the portfolio dataframe where the 'Symbol' column is empty
        portfolio_df = portfolio_df.dropna(subset=['Symbol'])

        # Ensure the ticker symbols are in uppercase
        portfolio_df['Symbol'] = portfolio_df['Symbol'].str.upper()

        # Merge the portfolio and stock prices dataframes on the 'Symbol' column
        merged_df = pd.merge(portfolio_df, stock_prices_df, how='left', left_on='Symbol', right_on='ticker')

        # Drop the 'symbol' column after merging
        merged_df = merged_df.drop(columns=['Symbol'])

        return merged_df

    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in

In [38]:
###################
#### DEBUGGING ####
###################

# Test the loading of the portfolio CSV file
portfolio_df = load_portfolio_csv('./data/portfolio.csv')

display(portfolio_df)

## Test the 'fetch_stock_price' for each stock in the portfolio
stock_prices = []
for ticker in portfolio_df['Symbol']:
    stock_price = fetch_stock_price(ticker)
    stock_prices.append(stock_price)

# Create a dataframe from the stock prices list
stock_prices_df = pd.DataFrame(stock_prices)

# Display the stock prices dataframe
display(stock_prices_df)

# Test Cleanse the stock prices dataframe
clean_stock_prices_df = cleanse_stock_prices(stock_prices_df)
display(clean_stock_prices_df)

# Merge the portfolio and stock prices data
merged_df = cleanse_and_merge_data(portfolio_df, clean_stock_prices_df)
display(merged_df)


,Symbol,Average_Cost,QTY
0,MSFT,200,300
1,AAPL,114,200
2,amzn,125,50
3,TSLA,900,100
4,NfLx,540,80
5,NVDA,450,50


,company_name,ticker,latest_open_price,latest_close_price,error
0,Microsoft Corporation,MSFT,431.654999,426.589996,None
1,Apple Inc.,AAPL,233.315002,233.399994,None
2,"Amazon.com, Inc.",amzn,189.580002,188.389999,None
3,"Tesla, Inc.",TSLA,269.876709,262.510010,None
4,"Netflix, Inc.",NfLx,758.679993,749.119995,None
5,NVIDIA Corporation,NVDA,143.029999,140.520004,None


,company_name,ticker,latest_open_price,latest_close_price
0,Microsoft Corporation,MSFT,431.65,426.59
1,Apple Inc.,AAPL,233.32,233.40
2,"Amazon.Com, Inc.",AMZN,189.58,188.39
3,"Tesla, Inc.",TSLA,269.88,262.51
4,"Netflix, Inc.",NFLX,758.68,749.12
5,Nvidia Corporation,NVDA,143.03,140.52


,Average_Cost,QTY,company_name,ticker,latest_open_price,latest_close_price
0,200,300,Microsoft Corporation,MSFT,431.65,426.59
1,114,200,Apple Inc.,AAPL,233.32,233.40
2,125,50,"Amazon.Com, Inc.",AMZN,189.58,188.39
3,900,100,"Tesla, Inc.",TSLA,269.88,262.51
4,540,80,"Netflix, Inc.",NFLX,758.68,749.12
5,450,50,Nvidia Corporation,NVDA,143.03,140.52


## **Serve Data**

**TODO**
- make sure the datasets match, names, tickers, values formatted the same
- upload the two files (export df CSV files) file search API
- assistant with files mounted -- code inter
    - assistnent with code interp and files mounted.
    - ask questions

### Upload Datasets

In [ ]:
from io import StringIO
from azure.ai.openai import OpenAIClient

def upload_dataframe_to_openai(df: pd.DataFrame, openai_client: OpenAIClient, file_name: str = "portfolio") -> dict:
    """
    Converts a DataFrame to CSV format and uploads it as a stream to Azure OpenAI File Search.

    Parameters:
    - df (pd.DataFrame): DataFrame to upload.
    - openai_client (OpenAIClient): Initialized Azure OpenAI client.
    - file_name (str, optional): Name for the file in Azure OpenAI. Defaults to "portfolio".

    Returns:
    - dict: Response from Azure OpenAI indicating upload success or error details.
    """
    # Convert DataFrame to CSV in-memory
    csv_stream = StringIO()
    df.to_csv(csv_stream, index=False)
    csv_stream.seek(0)  # Reset stream position to the beginning

    # Upload CSV stream to Azure OpenAI File Search
    response = openai_client.upload_file(
        file=csv_stream,
        file_name=file_name,
        purpose="file_search"
    )

    return response


### Create an Assistant